# Hole ring for laser-cut discs

Takes SVG files with round discs, scales each disc to a target diameter, adds a ring of small holes (for crochet) and lays the discs out on sheets no larger than the laser bed.

**Pipeline.** Load discs → scale and add holes → check clearances → pack onto sheets → export SVGs → preview.

Everything is measured from the file itself, so the source scale and units do not matter.

In [ ]:
# Install once if needed:
# !pip install svgelements numpy

## 1. Settings

In [ ]:
## @file disc_hole_ring.ipynb
#  @brief Add a crochet hole ring to laser-cut discs and pack them onto sheets.
#
#  Cell 1: settings. All sizes in mm.

## Input SVG files. Discs are read in this order.
INPUT_FILES = ["combined_2_dics.svg", "combined_1_disc.svg"]

# ---------------------------------------------------------------- disc and holes
## Final disc diameter.
DISC_DIAMETER_MM = 250
## Diameter of each crochet hole.
HOLE_DIAMETER_MM = 1.0
## Hole spacing, centre to centre, measured along the ring.
HOLE_PITCH_MM = 15.0
## Material left between the hole edge and the disc rim.
EDGE_MARGIN_MM = 5.3

# ---------------------------------------------------------------- sheets
## Laser bed size.
SHEET_W_MM, SHEET_H_MM = 700, 400
## Free border around the sheet.
SHEET_MARGIN_MM = 25
## Gap between neighbouring discs.
DISC_GAP_MM = 10
## Shrink the last sheet to its content (plus margin).
CROP_LAST_SHEET = True

# ---------------------------------------------------------------- checks and output
## Warn if a hole is closer than this to a cutout or the rim.
MIN_SAFE_GAP_MM = 3.0
## Stroke colour the laser software treats as "cut".
STROKE = "rgb(255,0,0)"
## Stroke width.
STROKE_MM = 0.1
## Prefix of the output file names.
OUT_PREFIX = "sheet"

## 2. Read the discs
The outer outline of a disc is the roundest, largest path. Every other path (the artwork cutouts) is assigned to the nearest disc.

In [ ]:
# Cell 2: read the discs
import math

import numpy as np
from svgelements import SVG, Path, Shape, Matrix

## A path counts as a circle when its radius spread is below this fraction.
ROUND_TOLERANCE = 0.002


def path_points(path, steps=40):
    """!
    @brief Sample points along every segment of a path.
    @param path   svgelements Path.
    @param steps  Samples per segment.
    @return (n, 2) array of x, y in user units.
    """
    pts = []
    for segment in path:
        for t in range(steps + 1):
            try:
                q = segment.point(t / steps)
            except Exception:              # Move segments have no points
                continue
            if q is not None:
                pts.append((q.x, q.y))
    return np.array(pts)


def load_paths(file):
    """!
    @brief Read every drawable shape of an SVG as a Path with transforms applied.
    @param file  SVG file name.
    @return List of svgelements Path.
    """
    paths = []
    for element in SVG.parse(file).elements():
        if isinstance(element, Shape) and not isinstance(element, SVG):
            p = Path(element)
            p.reify()                      # bake transforms into the coordinates
            if len(p) and len(path_points(p, 2)):
                paths.append(p)
    return paths


def circle_fit(pts):
    """!
    @brief Fit a circle using the bounding-box centre.
    @param pts  (n, 2) points.
    @return (cx, cy, r, roundness). roundness is the radius spread
            divided by the mean radius, near 0 for a true circle.
    """
    cx = (pts[:, 0].min() + pts[:, 0].max()) / 2
    cy = (pts[:, 1].min() + pts[:, 1].max()) / 2
    r = np.hypot(pts[:, 0] - cx, pts[:, 1] - cy)
    return cx, cy, r.mean(), r.std() / r.mean()


def bbox_centre(pts):
    """!
    @brief Centre of the bounding box of some points.
    @param pts  (n, 2) points.
    @return (x, y).
    """
    return ((pts[:, 0].min() + pts[:, 0].max()) / 2,
            (pts[:, 1].min() + pts[:, 1].max()) / 2)


def find_discs(paths):
    """!
    @brief Split paths into discs: outer circles plus the cutouts that belong to each.

    Outer circles are the round paths within 2 % of the largest round radius.
    Every other path goes to the disc whose centre is nearest.

    @param paths  List of Path from one file.
    @return List of dicts with keys "c" (centre), "r" (radius), "outline" and "cuts".
    """
    sampled = [(p, path_points(p)) for p in paths]
    fits = [circle_fit(pts) for _, pts in sampled]
    round_radii = [f[2] for f in fits if f[3] < ROUND_TOLERANCE]
    if not round_radii:
        raise ValueError("no round outline found in this file")
    r_max = max(round_radii)
    outer = [i for i, f in enumerate(fits) if f[3] < ROUND_TOLERANCE and f[2] > 0.98 * r_max]

    discs = [{"c": fits[i][:2], "r": fits[i][2], "outline": sampled[i][0], "cuts": []}
             for i in outer]
    for i, (p, pts) in enumerate(sampled):
        if i in outer:
            continue
        c = bbox_centre(pts)
        nearest = min(discs, key=lambda d: math.hypot(c[0] - d["c"][0], c[1] - d["c"][1]))
        nearest["cuts"].append(p)
    return discs


discs = []
for f in INPUT_FILES:
    found = find_discs(load_paths(f))
    print(f"{f}: {len(found)} disc(s), {[len(d['cuts']) for d in found]} cutout paths")
    discs += found
print("total discs:", len(discs))

## 3. Scale each disc and add the hole ring
The hole count is the ring circumference divided by the pitch, rounded to a whole number so the gaps stay even. The ring radius keeps `EDGE_MARGIN_MM` of material between hole edge and rim.

In [ ]:
# Cell 3: scale and hole ring

## Disc radius after scaling.
R_DISC = DISC_DIAMETER_MM / 2
## Hole radius.
R_HOLE = HOLE_DIAMETER_MM / 2
## Radius of the circle through the hole centres.
R_RING = R_DISC - EDGE_MARGIN_MM - R_HOLE
## Number of holes, so the actual pitch is as close as possible to HOLE_PITCH_MM.
N_HOLES = round(2 * math.pi * R_RING / HOLE_PITCH_MM)
print(f"ring radius {R_RING:.2f} mm, {N_HOLES} holes, "
      f"pitch {2 * math.pi * R_RING / N_HOLES:.2f} mm")


def to_disc_frame(path, disc):
    """!
    @brief Scale a path to the target disc size and move the disc centre to (0, 0).
    @param path  Path in source units.
    @param disc  Disc dict from find_discs.
    @return New Path in mm.
    """
    s = R_DISC / disc["r"]
    m = Matrix(s, 0, 0, s, -s * disc["c"][0], -s * disc["c"][1])
    q = Path(path) * m
    q.reify()
    return q


## Hole centres in the disc frame. The first hole sits at the top (-90°).
hole_centres = [(R_RING * math.cos(a), R_RING * math.sin(a))
                for a in (-math.pi / 2 + 2 * math.pi * k / N_HOLES for k in range(N_HOLES))]

for d in discs:
    d["outline_mm"] = to_disc_frame(d["outline"], d)
    d["cuts_mm"] = [to_disc_frame(p, d) for p in d["cuts"]]

## 4. Check clearances

In [ ]:
# Cell 4: clearance check

def hole_clearances(disc):
    """!
    @brief Smallest gaps from any hole edge to the artwork and to the rim.
    @param disc  Disc dict with "cuts_mm".
    @return (gap_to_cutout, gap_to_rim) in mm. gap_to_cutout is inf without cutouts.
    """
    gap_rim = R_DISC - R_RING - R_HOLE
    if not disc["cuts_mm"]:
        return math.inf, gap_rim
    pts = np.vstack([path_points(p, 60) for p in disc["cuts_mm"]])
    hc = np.array(hole_centres)
    nearest = np.hypot(pts[:, None, 0] - hc[None, :, 0], pts[:, None, 1] - hc[None, :, 1]).min()
    return nearest - R_HOLE, gap_rim


for i, d in enumerate(discs, 1):
    gap_cut, gap_rim = hole_clearances(d)
    flag = "" if min(gap_cut, gap_rim) >= MIN_SAFE_GAP_MM else "  <-- too close"
    print(f"disc {i}: hole to cutout {gap_cut:.2f} mm, hole to rim {gap_rim:.2f} mm{flag}")

## 5. Pack the discs onto sheets
Discs go left to right in one row. When the next disc would pass the sheet width, a new sheet starts. Each row is centred on its sheet.

In [ ]:
# Cell 5: pack onto sheets

def pack_rows(items, item_w, gap, margin, sheet_w):
    """!
    @brief Split items into sheets of one row each.
    @param items    Items to place, in order.
    @param item_w   Width of one item.
    @param gap      Gap between items.
    @param margin   Free border on each side.
    @param sheet_w  Sheet width.
    @return List of lists, one per sheet.
    """
    sheets, current, x = [], [], margin
    for item in items:
        if current and x + item_w + margin > sheet_w:
            sheets.append(current)
            current, x = [], margin
        current.append(item)
        x += item_w + gap
    if current:
        sheets.append(current)
    return sheets


if DISC_DIAMETER_MM + 2 * SHEET_MARGIN_MM > min(SHEET_W_MM, SHEET_H_MM):
    raise ValueError("Disc plus margins does not fit on the sheet")

sheets = pack_rows(discs, DISC_DIAMETER_MM, DISC_GAP_MM, SHEET_MARGIN_MM, SHEET_W_MM)
print([len(s) for s in sheets], "discs per sheet")

## 6. Write the SVGs

In [ ]:
# Cell 6: write the SVGs

def row_width(n):
    """!
    @brief Width of a row of n discs, without margins.
    @param n  Number of discs.
    @return Width in mm.
    """
    return n * DISC_DIAMETER_MM + (n - 1) * DISC_GAP_MM


def sheet_svg(group, w, h):
    """!
    @brief SVG text of one sheet: a centred row of discs with outline, cutouts and holes.
    @param group  Discs on this sheet.
    @param w, h   Sheet size in mm.
    @return SVG document as a string.
    """
    x0 = (w - row_width(len(group))) / 2 + R_DISC
    style = f'style="fill:none;stroke:{STROKE};stroke-width:{STROKE_MM}px;"'
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{w:g}mm" height="{h:g}mm" '
             f'viewBox="0 0 {w:g} {h:g}">']
    for i, d in enumerate(group):
        cx, cy = x0 + i * (DISC_DIAMETER_MM + DISC_GAP_MM), h / 2
        parts.append(f'<g id="disc{i + 1}" transform="translate({cx:.3f},{cy:.3f})" {style}>')
        parts.append(f'<path id="outline" d="{d["outline_mm"].d()}"/>')
        parts += [f'<path d="{p.d()}"/>' for p in d["cuts_mm"]]
        parts.append('<g id="holes">')
        parts += [f'<circle cx="{x:.3f}" cy="{y:.3f}" r="{R_HOLE:.3f}"/>' for x, y in hole_centres]
        parts += ['</g>', '</g>']
    parts.append('</svg>')
    return "\n".join(parts)


files = []
for k, group in enumerate(sheets, 1):
    w, h = SHEET_W_MM, SHEET_H_MM
    if CROP_LAST_SHEET and k == len(sheets):
        w = row_width(len(group)) + 2 * SHEET_MARGIN_MM
        h = DISC_DIAMETER_MM + 2 * SHEET_MARGIN_MM
    name = f"{OUT_PREFIX}{k}_{len(group)}disc_{DISC_DIAMETER_MM:g}mm_holes_{HOLE_DIAMETER_MM:g}mm.svg"
    with open(name, "w") as f:
        f.write(sheet_svg(group, w, h))
    files.append(name)
    print(f"{name}: {w:g} x {h:g} mm")

## 7. Preview

In [ ]:
# Cell 7: preview
from IPython.display import SVG as show_svg, display

for f in files:
    display(show_svg(filename=f))

## Notes
- Check one hole against a ruler after importing into your laser software.
- Put the holes before the outer outline in the cut order so the disc does not shift.
- 1 mm holes in Vivak can close slightly from heat. Test one cut first.